In [1]:
!pip install mitsuba

In [2]:
import mitsuba as mi
import matplotlib.pyplot as plt
import drjit as dr
import numpy as np
import math

In [3]:
mi.set_variant('cuda_ad_spectral')

In [9]:
class DispersiveDielectric(mi.BSDF):
    """
    Гладкий диэлектрик с дисперсией по Коши:
        n(λ) = n0 + B / λ²      (λ в мкм)
    """

    def __init__(self, props):
        mi.BSDF.__init__(self, props)

        self.int_ior = self._as_scalar(props.get('int_ior', 1.5046))
        self.ext_ior = self._as_scalar(props.get('ext_ior', 1.0))
        self.int_b   = self._as_scalar(props.get('int_b',   0.00420))
        self.ext_b   = self._as_scalar(props.get('ext_b',   0.0))

        reflection_flags = (
            mi.BSDFFlags.DeltaReflection
            | mi.BSDFFlags.FrontSide
            | mi.BSDFFlags.BackSide
        )
        transmission_flags = (
            mi.BSDFFlags.DeltaTransmission
            | mi.BSDFFlags.FrontSide
            | mi.BSDFFlags.BackSide
        )
        self.m_components = [reflection_flags, transmission_flags]
        self.m_flags = reflection_flags | transmission_flags

    # ---------------------------------------------------------------
    @staticmethod
    def _as_scalar(v):
        """
        Приводит значение к скалярному drjit.Float.
        - mi.Spectrum (Color3f / Spectrum) -> среднее по каналам
        - drjit-массив Float (например, si.wavelengths) -> оставляем как есть
        - Python float -> mi.Float
        """
        if isinstance(v, mi.Spectrum):
            return dr.mean(v)          # спектр -> скаляр
        elif dr.is_array_v(v):
            return v                   # уже Float-массив, не трогаем
        else:
            return mi.Float(v)         # Python float -> drjit.Float

    def _eval_eta(self, si):
        """Скалярный относительный показатель преломления eta = n_int / n_ext."""
        if mi.is_spectral:
            wl_um = si.wavelengths * 0.001          # nm -> µm, остаётся Float-массивом
            wl_sq = wl_um * wl_um
            int_ior = self.int_ior + self.int_b / wl_sq
            ext_ior = self.ext_ior + self.ext_b / wl_sq
        else:
            int_ior = self.int_ior
            ext_ior = self.ext_ior

        eta = int_ior / ext_ior
        return self._as_scalar(eta)     # гарантируем скалярный Float

    # ---------------------------------------------------------------
    def sample(self, ctx, si, sample1, sample2, active):
        eta = self._eval_eta(si)

        cos_theta_i = mi.Frame3f.cos_theta(si.wi)
        r_i, cos_theta_t, eta_it, eta_ti = mi.fresnel(cos_theta_i, eta)

        t_i = dr.maximum(1.0 - r_i, 0.0)
        selected_r = (sample1 <= r_i) & active

        bs = mi.BSDFSample3f()
        bs.pdf = dr.select(selected_r, r_i, t_i)
        bs.sampled_component = dr.select(selected_r, mi.UInt32(0), mi.UInt32(1))
        bs.sampled_type = dr.select(
            selected_r,
            mi.UInt32(+mi.BSDFFlags.DeltaReflection),
            mi.UInt32(+mi.BSDFFlags.DeltaTransmission),
        )
        bs.wo = dr.select(
            selected_r,
            mi.reflect(si.wi),
            mi.refract(si.wi, cos_theta_t, eta_ti),
        )
        bs.eta = dr.select(selected_r, 1.0, eta_it)

        value = dr.select(selected_r, 1.0, dr.sqr(eta_ti))
        return bs, value

    def eval(self, ctx, si, wo, active):
        return 0.0

    def pdf(self, ctx, si, wo, active):
        return 0.0

    def eval_pdf(self, ctx, si, wo, active):
        return 0.0, 0.0

    # ---------------------------------------------------------------
    def traverse(self, callback):
        callback.put_parameter('int_ior', self.int_ior, mi.ParamFlags.Differentiable)
        callback.put_parameter('ext_ior', self.ext_ior, mi.ParamFlags.Differentiable)
        callback.put_parameter('int_b',   self.int_b,   mi.ParamFlags.Differentiable)
        callback.put_parameter('ext_b',   self.ext_b,   mi.ParamFlags.Differentiable)

    def parameters_changed(self, keys):
        pass

    def to_string(self):
        return (f'DispersiveDielectric[int_ior={self.int_ior}, '
                f'ext_ior={self.ext_ior}, '
                f'int_b={self.int_b}, ext_b={self.ext_b}]')


mi.register_bsdf('dispersive_dielectric',
                 lambda props: DispersiveDielectric(props))
# ============================================================
# Тестовая сцена (Cornell box с двумя дисперсионными блоками)
# ============================================================
S = 100.0
P = mi.ScalarPoint3f
V = mi.ScalarVector3f
T = mi.ScalarTransform4f

scene_dict = {
    'type': 'scene',
    'integrator': {'type': 'path', 'max_depth': 8},
    
    # Камера смотрит сбоку, чтобы видеть и призму, и экран
    'sensor': {
        'type': 'perspective',
        'fov': 45,
        'to_world': T().look_at(
            P(0.5 * S, 1.0 * S, 3.5 * S),  # Позиция камеры
            P(1.0 * S, 0.5 * S, 0.0 * S),  # Куда смотрим (центр между призмой и экраном)
            P(0, 1, 0)
        ),
        'film': {'type': 'hdrfilm', 'width': 800, 'height': 600},
        'sampler': {'type': 'independent', 'sample_count': 512}, 
    },

    # Узкий прожектор, стреляющий вдоль оси X
    'light': {
        'type': 'spot',
        'to_world': T().look_at(
            P(-2.5 * S, 0.7 * S, 0.0),  # Позиция источника
            P(0.0, 0.5 * S, 0.0),       # Направление на призму
            P(0, 1, 0)
        ),
        'intensity': {'type': 'rgb', 'value': [100.0, 100.0, 100.0]}, # Нормальная яркость
        'cutoff_angle': 6.0, # Внешний угол пучка
        'beam_width': 3.0,   # Внутренний угол (мягкий край)
    },

    # Белый экран справа, на который упадет радуга
    'screen': {
        'type': 'rectangle',
        'to_world': (
            T().translate(P(2.0 * S, 0.5 * S, 0.0))
            .rotate(V(0, 1, 0), -90) # Поворачиваем лицом к призме
            .scale(V(1.5 * S, 1.5 * S, 1.0))
        ),
        'bsdf': {
            'type': 'diffuse',
            'reflectance': {'type': 'rgb', 'value': [0.9, 0.9, 0.9]}
        },
    },

    # Материал с сильной дисперсией
    'glass': {
        'type': 'dispersive_dielectric',
        'int_ior': 1.5046,   
        'ext_ior': 1.0,
        'int_b': 0.2,        # Сильная дисперсия
        'ext_b': 0.0,
    },

    # Призма в центре
    'prism': {
        'type': 'ply',
        'filename': 'prism.ply',
        'to_world': (
            T().translate(P(0.0, 0.5 * S, 0.0))
            .rotate(V(0, 1, 0), 45) # Поворот вокруг вертикальной оси
            .scale(V(0.4 * S, 0.4 * S, 0.4 * S))
        ),
        'bsdf': {'type': 'ref', 'id': 'glass'},
    },
}

scene = mi.load_dict(scene_dict)

RuntimeError: ​[properties.cpp:211] The property "int_ior" has the wrong type (expected <string>).

In [8]:
print("Рендеринг...")
image = mi.render(scene, spp=7000)

bitmap = mi.Bitmap(image)
bitmap = bitmap.convert(mi.Bitmap.PixelFormat.RGB, mi.Struct.Type.Float32, srgb_gamma=True)

plt.figure(figsize=(10, 10))
plt.imshow(bitmap)
plt.axis('off')
plt.show()

Рендеринг...


NameError: name 'scene' is not defined

In [ ]:
def spherical_positions(width: int, height: int, N: int, x: float, y: float, z: float, radius: float):
    sensors = []
    phi = np.pi * (3. - np.sqrt(5.))
    
    target = mi.ScalarPoint3f(x, y, z)

    for i in range(N):
        z_coord = 1 - (i / float(N - 1)) * 0.7
        radius_at_z = np.sqrt(1 - z_coord * z_coord)
        theta = phi * i
        
        px = np.cos(theta) * radius_at_z
        pz = z_coord
        py = np.sin(theta) * radius_at_z
        
        cam_pos = mi.ScalarPoint3f(
            x + px * radius,
            y + py * radius,
            z + pz * radius
        )
        up = (0.0, 0.0, 1.0) if abs(py) > 0.999 else (0.0, 1.0, 0.0)
        
        sensor = {
            "type": "perspective",
            "fov": 70,
            "to_world": mi.ScalarTransform4f().look_at(cam_pos, target, mi.ScalarPoint3f(*up)),
            'film': {
                'type': 'hdrfilm',
                'width': width,
                'height': height,
            }
        }
        sensors.append(sensor)
    return sensors

def use_spectrum(sensors: list):
    wavelengths = list(range(450, 651, 10))
    res = []

    for rgb_sensor in sensors:
        width, height = rgb_sensor['film']['width'], rgb_sensor['film']['height']
        wavelength_sensors = dict()
        wavelength_sensors['rgb'] = rgb_sensor

        for wavelength in wavelengths:
            film = {
                'type': 'specfilm',
                'width': width,
                'height': height,
                'channel1': {
                    'type': 'irregular',
                    'wavelengths': f'{wavelength}, {wavelength + 1}',
                    'values': '1.0, 1.0',
                },
            }
            wavelength_sensors[wavelength] = {**rgb_sensor, 'film': film}
        res.append(wavelength_sensors)
    return res

In [ ]:
rgb_cfgs = spherical_positions(640, 480, 2, 0, 0, 0, 0.3)

spectral_cfgs = use_spectrum(rgb_cfgs)

sensors = []
for group in spectral_cfgs:
    loaded_group = {k: mi.load_dict(v, parallel=False) for k, v in group.items()}
    sensors.append(loaded_group)

In [ ]:
global images
if globals().get('images'):
    print("Images is not empty. Type 'Y' to continue")
    if input().strip().upper() == 'Y':
        images = list()
if not globals().get('images'):
    images = list()
    print('rendering started')
    for i, sensorWavelength in enumerate(sensors):
        spectrum = dict()
        for wavelength, sensor in sensorWavelength.items():
            spectrum[wavelength] = mi.render(scene, sensor=sensor, spp=256)
            print(f'wavelength {wavelength} finished')
        images.append(spectrum)
        print(f'{i+1}/{len(sensors)}')
    print('rendering finished')

In [ ]:
def split_train_val(images, val_every=8):
    train_images = []
    val_images = []
    train_indices = []
    val_indices = []

    for i, spectrum in enumerate(images):
        if i % val_every == 0:
            val_images.append(spectrum)
            val_indices.append(i)
        else:
            train_images.append(spectrum)
            train_indices.append(i)

    return train_images, val_images, train_indices, val_indices


In [ ]:
train, val, train_idx, val_idx = split_train_val(images)


In [ ]:
import json
import os

import numpy as np
from PIL import Image


def save_dataset(images, sensor_cfgs, train_idx, val_idx, base_dir, wavelengths):
    width = sensor_cfgs[0]["film"]["width"]
    height = sensor_cfgs[0]["film"]["height"]
    fov = float(sensor_cfgs[0]["fov"])
    fl = 0.5 * width / math.tan(math.radians(fov) * 0.5)
    meta = {
        "camera_angle_x": math.radians(fov),
        "fl_x": fl,
        "fl_y": fl,
        "cx": 0.5 * width,
        "cy": 0.5 * height,
        "w": width,
        "h": height,
        "camera_model": "OPENCV",
        "k1": 0.0,
        "k2": 0.0,
        "p1": 0.0,
        "p2": 0.0,
        "wavelengths": [int(w) for w in wavelengths],
    }

    os.makedirs(os.path.join(base_dir, "train"), exist_ok=True)
    os.makedirs(os.path.join(base_dir, "eval"), exist_ok=True)

    val_idx = set(val_idx)
    frames = []

    for idx, (image, cfg) in enumerate(zip(images, sensor_cfgs)):
        split = "eval" if idx in val_idx else "train"
        name = f"r_{idx}"

        rgb = np.clip(np.asarray(image["rgb"])[..., :3], 0.0, 1.0)
        rgb = np.where(rgb < 0.0031308, 12.92 * rgb, 1.055 * np.power(rgb, 1 / 2.4) - 0.055)
        Image.fromarray((rgb * 255).round().astype(np.uint8)).save(
            os.path.join(base_dir, split, f"{name}.png")
        )

        cube = np.zeros((height, width, len(wavelengths)), dtype=np.float32)
        for i, wavelength in enumerate(wavelengths):
            band = np.asarray(image[wavelength])
            if band.ndim == 3:
                band = band[..., 0]
            cube[..., i] = np.clip(band, 0.0, 1.0)
        np.save(os.path.join(base_dir, split, f"{name}.npy"), cube)

        c2w = np.array(cfg["to_world"].matrix, dtype=np.float64)
        c2w[:3, 0] *= -1
        c2w[:3, 2] *= -1

        frames.append({
            "file_path": f"{split}/{name}.png",
            "rotation": 0.0,
            "transform_matrix": c2w.tolist(),
            "hyperspectral_file_path": f"{split}/{name}.npy",
        })

    train_frames = [frames[i] for i in train_idx]
    val_frames = [frames[i] for i in val_idx]

    for name, split_frames in (("train", train_frames), ("val", val_frames), ("test", frames)):
        with open(os.path.join(base_dir, f"transforms_{name}.json"), "w") as f:
            json.dump({**meta, "frames": split_frames}, f, indent=4)

    with open(os.path.join(base_dir, "transforms.json"), "w") as f:
        json.dump({**meta, "frames": frames}, f, indent=4)


In [ ]:
wavelengths = sorted(k for k in images[0] if k != "rgb")

save_dataset(images, rgb_cfgs, train_idx, val_idx, "data/my_scene", wavelengths)


In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt

def show_images(images: dict, gamma: float = 2.2, cols: int = 4, cmap=None):
    items = list(images.items())
    n = len(items)
    cols = min(cols, n)
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows), squeeze=False)

    for ax, (title, image) in zip(axes.flat, items):
        img = np.asarray(image, dtype=np.float64)

        img = np.clip(img, 0.0, 1.0)

        img_g = np.power(img, 1.0 / gamma)

        ax.imshow(img_g, cmap=cmap, vmin=0.0, vmax=1.0)
        ax.set_title(title)
        ax.axis("off")

    for ax in axes.flat[n:]:
        ax.axis("off")

    fig.tight_layout()
    plt.show()

In [ ]:
show_images(images[0])